<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

----------

<h1 align="left"> Build a Research Paper Assistant using RAG</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

----------


#**Objective:**

In this lab, you’ll build a simple Retrieval-Augmented Generation (RAG) system that can answer questions about research paper abstracts from the arXiv dataset.

You’ll:

1. Load a small dataset of paper abstracts.
2. Create embeddings for each abstract.
3. Store and search these embeddings using FAISS.









In [2]:

!pip install -q sentence-transformers transformers faiss-cpu datasets==2.14.5

**Import Libraries**

In [3]:
#  required libraries :

from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import pandas as pd
from datasets import load_dataset

**Load and Explore Research Data**

**Scenario**

You are building an assistant for AIResearchHub, a company that wants to summarize or answer questions about new computer-science papers.

In [4]:
from datasets import load_dataset
import pandas as pd

# Use scientific_papers (arXiv split) and take a small subset
dataset = load_dataset("armanc/scientific_papers", "arxiv", split="train[:100]")

df = pd.DataFrame(dataset) [["abstract"]]

df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

,abstract
0,additive models play an important role in sem...
1,"we have studied the leptonic decay @xmath0 , ..."
2,"in 84 , 258 ( 2000 ) , mateos conjectured tha..."
3,the effect of a random phase diffuser on fluc...
4,with a special intention of clarifying the un...


In [6]:
print("Number of papers:", len(df))
print("\nSample abstract:\n")
print(df['abstract'][0][:500], "...")


Number of papers: 100

Sample abstract:

 additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for additive models . 
 these learning rates compare favourably in particular in high dimensions to recent results on optimal learning rates for purely nonparametric regularized kernel based quantile regression using the gaussian radial basis function kernel , provided the assumption of an additive model is valid . 
 additionally , a concrete example is pr ...


**Create Document Embeddings**

In [7]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


documents = df['abstract'].tolist()
doc_embeddings = embedding_model.encode(documents, convert_to_numpy=True, normalize_embeddings=True)
print("Embeddings shape:", doc_embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (100, 384)


**Build a FAISS Index for Retrieval**





In [8]:
dimension = doc_embeddings.shape[1] #the size
index = faiss.IndexFlatL2(dimension)

index.add(doc_embeddings)
print(f"FAISS index contains {index.ntotal} abstracts.")


FAISS index contains 100 abstracts.


**Retrieval Function**

In [18]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# TODO: Write the retrieval function
# HINT: Use index.search() and retrieve top_k most similar abstracts
def retrieve(query, top_k=3):
    query_embedding = embedder.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return df.iloc[indices[0]]["abstract"].tolist()

**Test**

In [19]:
query = "What are recent advancements in reinforcement learning?"
results = retrieve(query)

for i, abs_text in enumerate(results, start=1):
    print(f"\nAbstract {i}:\n{abs_text[:400]}...")


Abstract 1:
 synaptic memory is considered to be the main element responsible for learning and cognition in humans . 
 although traditionally non - volatile long - term plasticity changes have been implemented in nanoelectronic synapses for neuromorphic applications , recent studies in neuroscience have revealed that biological synapses undergo meta - stable volatile strengthening followed by a long - term st...

Abstract 2:
 additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for additive models . 
 these learning rates compare favourably in particular in high dimensions to recent results on optimal learning rates for purely nonparametric regularized kernel based quantile regression using the gaussian radial basis function kernel...

Abstract 3:
 we propose a tuner , suitable for adaptive control and ( in its discrete - time version ) adaptive filtering applications , that sets the second derivat

** *italicized text*Add a Generation Model (FLAN-T5)**

In [21]:
from transformers import pipeline


generator = pipeline("text2text-generation", model="google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


** *italicized text*Define RAG Pipeline**

In [23]:
  # TODO: Implement rag_pipeline(query)
  # HINT: Prompt format example:
# "Context: <retrieved abstracts>\nQuestion: <query>\nAnswer:"
def rag_pipeline(query):
    retrieved_docs = retrieve(query)

    context = " ".join(retrieved_docs)
    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

    response = generator(prompt, max_length=200)[0] ["generated_text"]


    print("🔍 Retrieved Abstracts:")
    for i, doc in enumerate(retrieved_docs, start=1):
        print(f" - Abstract {i}: {doc[:120]}...")

    print("\n💬 Generated Answer:\n", response)





**Ask A question**

In [ ]:
rag_pipeline("What are the applications of graph neural networks?")


Token indices sequence length is longer than the specified maximum sequence length for this model (793 > 512). Running this sequence through the model will result in indexing errors
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
